In [1]:
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree # Change to the project directory
%env PATH=$HOME/.local/bin:$PATH
    
import json
import os
import requests
import numpy as np
import cv2
from PIL import Image
from pymongo import MongoClient
# from mmdet.apis import init_detector, inference_detector
# from mmdet.utils import register_all_modules
from data_processing.divide_photos import divide_tablet_photo
from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.data_source import LocalDataSource
from sign_alignment.visualizer import BboxVisualizer, ColorConfig

# import signs_alignment as sa
import os
import torch
from dotenv import load_dotenv

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
# # temporal change
# CHECKPOINT_FILE = os.path.expanduser("~/epoch_1000.pth")
# ANNOTATIONS_DIR = os.path.expanduser("~/filtered_annotations")
# # ---
SCORE_THRESHOLD = 0.5
OUTPUT_DIR = "alignment_results"
SAMPLE_LIMIT = 5  # number of samples to process

load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI', 'YOUR_MONGODB_URI')


[Errno 2] No such file or directory: '/home/jebediahc/erc-src/cuneiform-ocr-sign-alignment-worktree # Change to the project directory'
/home/jebediahc/erc-src/cuneiform-ocr-sign-alignment-worktree
env: PATH=$HOME/.local/bin:$PATH
classes setting in current coco.py: ['TU', '|U.GUD|', 'TUM', 'LA', 'TA', 'GAR', 'GAL', 'I', 'TI', 'LI', 'ZA', 'A', 'DI', 'MI', 'RI', 'IŠ', 'BA', 'LU', 'TE', 'DA', '|GUD×KUR|', 'MA', 'E₂', 'DIŠ', 'MU', 'DU', 'ŠU₂', 'EN', 'KUL', 'SI', '|I.A|', 'HI', 'MUŠ₃', 'AN', 'NA', 'BAD', 'AMAR', 'UD', 'UnclearSign', '|HI×BAD|', '|UD×(U.U.U)|', 'AB', 'AK', 'LUGAL', 'DIN', 'KI', 'DUN₃@g', 'KU₃', 'AŠ', 'IGI', 'U₂', 'ŠA₃', 'BI', 'GUR', 'ŠE', 'ZI', 'GA', 'SILA₃', 'ŠID', '|SAL.TUG₂|', 'SU', 'KAK', 'MAŠ', 'TUR', 'ŠEŠ', 'LU₂', 'IA₂', 'UR', 'KAL', '|ŠEŠ.KI|', 'ZU', 'ŠU', 'NE', 'IM', 'RA', '|U.U|', 'ZAG', '|DIŠ.DIŠ.DIŠ|', 'GA₂', 'IN', 'KIN', 'TAR', 'MAH', 'LAL', 'KID', 'GABA', 'KA', 'RU', 'ŠA', '|HI×NUN|', 'ME', 'BU', 'NI', 'IG', 'MES', 'PA', 'SAG', 'U', 'E', 'GUM', 'GIŠ', '|U.KA|', 

In [2]:
from sign_alignment.pipeline import CropContext, PipelineConfig, DEBUG_STEPS, Runner

model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device='auto'
)
tablet_detector = TabletImageDetector(
    model_config=model_config,
    score_threshold=SCORE_THRESHOLD,
    keep_crops=True
)

crop_context = CropContext(
    config=PipelineConfig(
        model_config=model_config,
        tablet_detector=tablet_detector,
        local_source=LocalDataSource(ANNOTATIONS_DIR),
        color_config=ColorConfig,
        output_dir=OUTPUT_DIR,
    )
)

runner = Runner(
    context=crop_context,
    steps=DEBUG_STEPS
)

Initializing detector, loading model...
Using device: cpu
Loads checkpoint by local backend from path: /home/jebediahc/erc-work-data/retrained_models/detr-173/epoch_1000.pth
Found 947 fragments with both image and annotation


In [3]:
import sign_alignment.pipeline as pp

runner.choose_sample(9)  # index 0 = NBC.4020, index 9 = HS.2086
runner.run_single_step(pp.step_load_data)

Processing sample: HS.2086
Step: Load Data
Ground truth boxes: 397


In [6]:
runner.run_single_step(pp.step_show_ground_truth)

Step: Show Ground Truth
✓ Saved to: /home/jebediahc/erc-src/cuneiform-ocr-sign-alignment-worktree/alignment_results/debug_HS.2086_gt.jpg


In [7]:
# get sign text (gt) from text.lines with broken sign filtering
runner.run_single_step(pp.step_load_sign_text)

Step: Load Sign Text from API
  Text lines: 47, total signs: 498
  Unfiltered: 506 signs, broken signs removed: 8


In [ ]:
# detect signs (full image + chosen exp_image crop)
runner.run_single_step(pp.step_detect_signs)

In [ ]:
# transform GT boxes into exp_image coordinates and visualize
runner.run_single_step(pp.step_transform_gt_to_exp)

In [ ]:
# compute average detection box dimensions
runner.run_single_step(pp.step_compute_statistics)

In [ ]:
# create detection and text sub-tablets
runner.run_single_step(pp.step_create_subtablets)

In [ ]:
# DBSCAN row detection on detection sub-tablet (also reports text sub-tablet rows)
runner.run_single_step(pp.step_detect_rows)

In [ ]:
# DP row matching between detection and text sub-tablets
runner.run_single_step(pp.step_match_rows)

In [ ]:
# visualize detection rows with D# / D#→R# labels
runner.run_single_step(pp.step_visualize_detection_rows)

In [ ]:
# within-row sign matching for each matched row pair
runner.run_single_step(pp.step_match_signs_in_rows)

In [ ]:
# align text rows onto detection rows using regression baselines
runner.run_single_step(pp.step_align_text_rows)

In [ ]:
# build sub_tablet_optim from aligned text boxes
runner.run_single_step(pp.step_create_optim_subtablet)

In [ ]:
# build sign match info; draw text mapping, side-by-side composite, alignment diagnostic
runner.run_single_step(pp.step_build_sign_match_info)

In [ ]:
# position offset analysis for matched signs
runner.run_single_step(pp.step_offset_analysis)

In [ ]:
# create PSR optimizer and plot characteristic loss curves
runner.run_single_step(pp.step_create_psr_optimizer)

In [ ]:
# run PSR optimization (produces sub_tablet_final)
runner.run_single_step(pp.step_run_psr_optimization)

In [ ]:
# optimization loss history
runner.run_single_step(pp.step_plot_loss_history)

In [ ]:
# 2x2 results comparison: coarse aligned, final optimized, det+final overlay, gt+final overlay
runner.run_single_step(pp.step_results_comparison)

In [ ]:
# analyze parameter changes between coarse-aligned and final optimized
runner.run_single_step(pp.step_param_changes)